<span style="font-size: 24px;">Ejercicio 1. Creación de una bases de datos de películas extraídas de un API </span>

In [ ]:
# Importo las librerías que voy a necesitar

import requests  # Para llamar a las APIs
import pandas as pd  # Para poder convertir los datos en df
import mysql.connector  # Para conectar Python con MySQL
from mysql.connector import Error  # Para importar los errores de SQL
import os   # Para crear variables de entorno y poder mantener la contraseña secreta
from dotenv import load_dotenv  # Para leer el archivo .env de la variable de entorno
load_dotenv()  # Carga el contenido del .env
password_sql = os.getenv("pass_sql") # Para acceder a la contraseña que hemos indicado en el .env
import numpy as np  # Para poder hacer el clean

<span style="font-size: 20px;">Fase 1: Extracción de datos de películas</span>
 
El objetivo es extraer 100 películas de esta API utilizando el siguiente endpoint:
https://beta.adalab.es/resources/apis/pelis/pelis.json
Datos a extraer: título, año de lanzamiento, duración (en minutos), género, contenidon para adultos (sí o no).

In [38]:
url = "https://beta.adalab.es/resources/apis/pelis/pelis.json"

datos = requests.get(url)   # Hacemos la petición a la API

In [39]:
datos.status_code  # Confirmamos que la petición ha sido exitosa

200

In [ ]:
datos_pelis = datos.json()  # Pasamos los datos a formato json para poder leerlos bien en Python
df_pelis = pd.DataFrame(datos_pelis)  # Convertimos esos datos en tabla (DataFrame) con ayuda de Pandas
df_pelis

,id,titulo,año,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,False,"[es, en]"
1,2,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
2,3,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
3,4,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
4,5,The Dark Knight,2008,152,Acción,False,"[es, en]"
...,...,...,...,...,...,...,...
95,96,La vita è bella,1997,116,Drama,False,"[es, en, it]"
96,97,Requiem for a Dream,2000,102,Drama,True,"[es, en]"
97,98,Memento,2000,113,Thriller,True,"[es, en]"
98,99,Eternal Sunshine of the Spotless Mind,2004,108,Drama,False,"[es, en]"


<span style="font-size: 20px;">Fase 2: Creación de la Base de Datos</span>

Podemos elegir entre crear la base de datos en MySQLWorkbench o en Python, yo voy a elegir utilizar Python.

In [ ]:
# Nos conectamos con MySQL

try:
    connection = mysql.connector.connect (
    host = "127.0.0.1", 
    user = "root",
    password = password_sql, 
    )
    print ("Conexion exitosa")
except Error as e:
    print ("Ha ocurrido un error: {e}")

Conexion exitosa


In [ ]:
# Vamos a crear el cursor y la base de datos

try:
    cursor = connection.cursor()
    query_crear_bbdd = "CREATE DATABASE IF NOT EXISTS pelis"
    cursor.execute(query_crear_bbdd)
    print("Query existosa")
except Error as e:
    print(e)

Query existosa


In [ ]:
# Queremos saber qué tipo de datos tiene la tabla importada para poder crear nuestra tabla en la base de datos

df_pelis.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          100 non-null    int64 
 1   titulo      100 non-null    str   
 2   año         100 non-null    int64 
 3   duracion    100 non-null    int64 
 4   genero      100 non-null    str   
 5   adultos     100 non-null    bool  
 6   subtitulos  100 non-null    object
dtypes: bool(1), int64(3), object(1), str(2)
memory usage: 4.9+ KB


In [ ]:
# Le pedimos que use la base de datos que acabamos de crear y creamos la tabla

cursor.execute("use pelis")
query_borrar = '''drop table pelis'''
cursor.execute(query_borrar)
connection.commit()
query_crear_tabla = '''CREATE TABLE pelis (
                        id INT PRIMARY KEY AUTO_INCREMENT, 
                        titulo VARCHAR(200) NOT NULL,
                        año YEAR,
                        duracion INT,
                        genero VARCHAR(100),
                        adultos BOOL,
                        subtitulos VARCHAR(50)
                                               );'''
cursor.execute(query_crear_tabla)
print ("Tabla creada correctamente")

Tabla creada correctamente


<span style="font-size: 20px;">Fase 3: Inserción de los Datos en la Base de Datos</span>

En este apartado vamos a realizar la inserción de los datos extraídos en las tablas creadas en nuestra base de datos MySQL.

In [45]:
# Vamos a ver si hay alguna celda vacía

df_pelis.isnull().sum()

id            0
titulo        0
año           0
duracion      0
genero        0
adultos       0
subtitulos    0
dtype: int64

In [46]:
# Necesito convertir subtitulos en un string, porque ahora es una lista

df_pelis['subtitulos'] = df_pelis['subtitulos'].astype(str)

In [48]:
# Añadimos los datos que hemos importado de la API a la tabla de nuestra base de datos

try:
    query_insertar = '''INSERT INTO pelis (id, titulo, año, duracion, genero, adultos, subtitulos)
                        VALUES (%s, %s, %s, %s, %s, %s, %s)'''
    values = df_pelis.values.tolist() 
    cursor.executemany(query_insertar, values)
    connection.commit()
    print(f"Se han insertado nuevos {cursor.rowcount} valores")
except Error as e:
    print(e)

Se han insertado nuevos 100 valores
